# Pandas Merge, Concat ve Reshape

Gerçek projelerde veriler çoğu zaman tek tabloda gelmez. Müşteri tablosu, sipariş tablosu, ürün tablosu gibi parçaları birleştirmek gerekir.

Bu notebook'ta `merge`, `concat`, `pivot_table` ve `melt` konularını öğreneceğiz.


## Konu Dokümantasyonu

Veri birleştirme ve yeniden şekillendirme veri manipülasyonunun önemli parçalarıdır.

* `merge`: Ortak anahtar üzerinden tabloları birleştirir.
* `concat`: Tabloları alt alta veya yan yana ekler.
* `pivot_table`: Uzun veriyi özet tabloya çevirir.
* `melt`: Geniş veriyi uzun formata çevirir.

Makine öğrenmesi ve raporlama süreçlerinde veriyi doğru formata getirmek çoğu zaman modelden daha önemli bir adımdır.


## Kolay Seviye

`merge`, SQL join mantığına benzer. İki tablo ortak bir anahtar üzerinden birleştirilir.


In [ ]:
import pandas as pd

siparisler = pd.DataFrame({
    "siparis_id": [1, 2, 3],
    "musteri_id": [101, 102, 101],
    "tutar": [1200, 950, 1800],
})

musteriler = pd.DataFrame({
    "musteri_id": [101, 102],
    "musteri": ["Ali", "Ayşe"],
})

sonuc = siparisler.merge(musteriler, on="musteri_id", how="left")
print(sonuc)


In [ ]:
import pandas as pd

ocak = pd.DataFrame({"ay": ["Ocak", "Ocak"], "satis": [1200, 950]})
subat = pd.DataFrame({"ay": ["Şubat", "Şubat"], "satis": [1800, 2100]})

birlesik = pd.concat([ocak, subat], ignore_index=True)
# ignore_index=True yeni düzenli indeks oluşturur

print(birlesik)


## Orta Seviye

`pivot_table`, gruplama sonucunu tablo formunda görmeyi sağlar. Satır ve sütun eksenlerine değişkenler yerleştirilir.


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "sehir": ["İstanbul", "İstanbul", "Ankara", "Ankara"],
    "kategori": ["Elektronik", "Mobilya", "Elektronik", "Mobilya"],
    "satis": [30000, 7500, 20000, 9000],
})

pivot = df.pivot_table(
    values="satis",
    index="sehir",
    columns="kategori",
    aggfunc="sum",
    fill_value=0,
)

print(pivot)


In [ ]:
import pandas as pd

genis_df = pd.DataFrame({
    "urun": ["Laptop", "Telefon"],
    "Ocak": [30000, 20000],
    "Şubat": [35000, 24000],
})

uzun_df = genis_df.melt(
    id_vars="urun",
    var_name="ay",
    value_name="satis",
)

print(uzun_df)


## İleri Seviye

Birden fazla tabloyu birleştirirken hangi join türünün kullanılacağı önemlidir.

* `left`: Sol tablodaki tüm satırları korur.
* `inner`: Sadece iki tabloda da eşleşenleri alır.
* `outer`: Tüm eşleşen ve eşleşmeyen satırları alır.

Veri kaybı yaşamamak için merge sonrası satır sayısı ve eksik değerler kontrol edilmelidir.


In [ ]:
import pandas as pd

siparisler = pd.DataFrame({
    "siparis_id": [1, 2, 3, 4],
    "urun_id": [10, 20, 30, 40],
    "adet": [1, 2, 1, 3],
})

urunler = pd.DataFrame({
    "urun_id": [10, 20, 30],
    "urun": ["Laptop", "Telefon", "Tablet"],
    "fiyat": [30000, 20000, 12000],
})

sonuc = siparisler.merge(urunler, on="urun_id", how="left")
sonuc["tutar"] = sonuc["adet"] * sonuc["fiyat"]

print(sonuc)
print("Eksik ürün bilgisi:", sonuc["urun"].isna().sum())


## Mini Veri Bilimi Uygulaması

Sipariş ve ürün tablolarını birleştirip kategori bazında toplam ciro hesaplayalım.


In [ ]:
import pandas as pd

siparisler = pd.DataFrame({
    "siparis_id": [1, 2, 3, 4],
    "urun_id": [10, 20, 10, 30],
    "adet": [1, 2, 1, 3],
})

urunler = pd.DataFrame({
    "urun_id": [10, 20, 30],
    "urun": ["Laptop", "Telefon", "Kulaklık"],
    "kategori": ["Elektronik", "Elektronik", "Aksesuar"],
    "fiyat": [30000, 20000, 2500],
})

df = siparisler.merge(urunler, on="urun_id", how="left")
df["ciro"] = df["adet"] * df["fiyat"]

kategori_ciro = df.groupby("kategori")["ciro"].sum().sort_values(ascending=False)
print(kategori_ciro)
